In [1]:
import os
import sys
import numpy as np
import pandas as pd
import seaborn as sns
import matplotlib.pyplot as plt
from datetime import datetime
import json
from pyspark.sql import SparkSession
from pyspark.sql import functions as F
import matplotlib.pyplot as plt
import pyspark
from pyspark.sql.types import StructType, StructField, StringType, IntegerType, DoubleType, LongType

In [2]:
pyspark_home = os.path.dirname(pyspark.__file__)

os.environ['SPARK_HOME'] = pyspark_home
os.environ['PATH'] += os.pathsep + os.path.join(pyspark_home, 'bin')
# Manually point to your Java and Hadoop locations
os.environ['JAVA_HOME'] = r"C:\Program Files\Eclipse Adoptium\jdk-17.0.18.8-hotspot"
os.environ['HADOOP_HOME'] = r"C:\hadoop"

# Force the 'bin' folders into the session path
os.environ['PATH'] += os.pathsep + os.path.join(os.environ['JAVA_HOME'], 'bin')
os.environ['PATH'] += os.pathsep + os.path.join(os.environ['HADOOP_HOME'], 'bin')

# Tell Spark exactly which python to use
os.environ['PYSPARK_PYTHON'] = sys.executable
os.environ['PYSPARK_DRIVER_PYTHON'] = sys.executable



if "spark" in globals():
    spark.stop()
spark = SparkSession.builder.appName("CarValidation").config("spark.sql.ansi.enabled", "false").getOrCreate()
sc = spark.sparkContext

In [3]:
df = spark.read.csv("data/raw/craigslistVehicles.csv", header=True, inferSchema=False, multiLine=True, escape='"')

# Print the top 10 rows
df.show(5)

actual_dtypes = dict(df.dtypes)
print("Actual Data Types:")
for col, dtype in actual_dtypes.items():
    print(f"{col}: {dtype}")
    
validation_results = []
metadata = {
            'total_rows': df.count(),
            'total_columns': len(df.columns),
            'scan_time': datetime.now().strftime("%Y-%m-%d %H:%M:%S")
}


+--------------------+----------------+--------------------+-----+----+------------+--------------------+---------+-----------+------+--------+------------+------------+-----------------+-----+--------+-----+-----------+--------------------+--------------------+---------+----------+
|                 url|            city|            city_url|price|year|manufacturer|                make|condition|  cylinders|  fuel|odometer|title_status|transmission|              VIN|drive|    size| type|paint_color|           image_url|                desc|      lat|      long|
+--------------------+----------------+--------------------+-----+----+------------+--------------------+---------+-----------+------+--------+------------+------------+-----------------+-----+--------+-----+-----------+--------------------+--------------------+---------+----------+
|https://grandrapi...|grand rapids, MI|https://grandrapi...| 1500|2006|    cadillac|                 cts|     good|6 cylinders|   gas|  236000|     

In [4]:
def add_to_report(check_type, passed, issues):
    """Internal helper to standardize report entries."""
    report = {
        'timestamp': datetime.now().strftime("%Y-%m-%d %H:%M:%S"),
        'check_type': check_type,
        'passed': passed,
        'issues': issues
    }
    validation_results.append(report)

In [5]:
def check_completeness(df):
    # 1. Read everything as String to prevent casting crashes
    # We use 'multiLine' because 'desc' columns often have newlines
    # We use 'escape' to handle quotes within descriptions
    issues = []
    passed = True
    
    total_rows = df.count()
    print(f"Total Rows: {total_rows}")

    # 2. Define the columns we EXPECT to be numeric
    numeric_cols = ["price", "year", "odometer", "lat", "long"]
    
    # 3. Optimized Single-Pass Validation
    # We calculate Nulls, NaNs, and Type-Casting Failures in one go
    validation_logic = []
    
    for c in df.columns:
        # 1. Define what "Missing" looks like in this dataset
        # We check for actual Nulls, the string "NULL", or empty strings
        is_null = F.col(c).isNull() | (F.col(c) == "NULL") | (F.col(c) == "")
        validation_logic.append(F.count(F.when(is_null, c)).alias(f"{c}_missing"))
        
        # 2. Check for Type Errors (if column is supposed to be numeric)
        if c in numeric_cols:
            # We attempt to cast. If it results in NULL but wasn't originally NULL, it's a type error.
            is_invalid_type = F.col(c).isNotNull() & (F.col(c) != "") & F.col(c).cast("double").isNull()
            validation_logic.append(F.count(F.when(is_invalid_type, c)).alias(f"{c}_type_error"))

    print("Analyzing Completeness")
    # Single action: one scan through the whole file
    results = df.select(validation_logic).collect()[0].asDict()

    
    for c in df.columns:
        missing = results.get(f"{c}_missing", 0)
        missing_pct = (missing / total_rows) * 100
        if missing_pct > 0:
            passed = False
            issues.append(f"Column '{c}': {missing_pct:.2f}% missing")
    add_to_report("Completeness", passed, issues)
        

check_completeness(df)

Total Rows: 525839
Analyzing Completeness


In [6]:
# 1. Define your Expected Schema
schema = StructType([
    StructField("url", StringType(), True),
    StructField("city", StringType(), True),
    StructField("city_url", StringType(), True),
    StructField("price", LongType(), True),
    StructField("year", IntegerType(), True),
    StructField("manufacturer", StringType(), True),
    StructField("make", StringType(), True),
    StructField("condition", StringType(), True),
    StructField("cylinders", StringType(), True),
    StructField("fuel", StringType(), True),
    StructField("odometer", DoubleType(), True),
    StructField("title_status", StringType(), True),
    StructField("transmission", StringType(), True),
    StructField("VIN", StringType(), True),
    StructField("drive", StringType(), True),
    StructField("size", StringType(), True),
    StructField("type", StringType(), True),
    StructField("paint_color", StringType(), True),
    StructField("image_url", StringType(), True),
    StructField("desc", StringType(), True),
    StructField("lat", DoubleType(), True),
    StructField("long", DoubleType(), True)
])

def check_schema_and_types(df, expected_schema):
    print("\n--- Metadata & Type Conformance Check ---")
    issues= []
    passed= True

    if df.columns != expected_schema.names:
        passed = False

        missing_cols = set(expected_schema.names) - set(df.columns)
        if missing_cols:
            issues.append(f"Missing columns: {missing_cols}")

        extra_cols = set(df.columns) - set(expected_schema.names)
        if extra_cols:
            issues.append(f"Extra columns (not in schema): {extra_cols}")

    # Optimized Type Check using expr()
    validation_exprs = []
    numeric_fields = [f for f in expected_schema.fields if not isinstance(f.dataType, StringType)]
    
    for field in numeric_fields:
        col_name = field.name
        data_type = field.dataType.simpleString() # e.g., 'bigint' or 'double'
        
        # We use F.expr to call the SQL version of try_cast
        # If the cast fails (like for '3009548743' to INT), it returns NULL
        # but does NOT crash the job.
        invalid_mask = F.col(col_name).isNotNull() & \
                       (F.col(col_name) != "") & \
                       F.expr(f"try_cast({col_name} as {data_type})").isNull()
        
        validation_exprs.append(F.count(F.when(invalid_mask, True)).alias(col_name))

    results = df.select(validation_exprs).collect()[0].asDict()
    
    for col, error_count in results.items():
        if error_count != 0:
            issues.append(
                f"Column '{col}': {error_count} rows fail casting (likely overflow or text)."
            )
            passed = False
    add_to_report("Schema & Types", passed, issues)

check_schema_and_types(df, schema)

df_casted = df
for field in schema.fields:
    col_name = field.name
    target_type = field.dataType
    # Apply the cast from the schema blueprint to the column
    df_casted = df_casted.withColumn(col_name, F.col(col_name).cast(target_type))

# Now df_casted has the actual Longs, Integers, and Doubles.
# Let's verify:
df_casted.printSchema()


--- Metadata & Type Conformance Check ---
root
 |-- url: string (nullable = true)
 |-- city: string (nullable = true)
 |-- city_url: string (nullable = true)
 |-- price: long (nullable = true)
 |-- year: integer (nullable = true)
 |-- manufacturer: string (nullable = true)
 |-- make: string (nullable = true)
 |-- condition: string (nullable = true)
 |-- cylinders: string (nullable = true)
 |-- fuel: string (nullable = true)
 |-- odometer: double (nullable = true)
 |-- title_status: string (nullable = true)
 |-- transmission: string (nullable = true)
 |-- VIN: string (nullable = true)
 |-- drive: string (nullable = true)
 |-- size: string (nullable = true)
 |-- type: string (nullable = true)
 |-- paint_color: string (nullable = true)
 |-- image_url: string (nullable = true)
 |-- desc: string (nullable = true)
 |-- lat: double (nullable = true)
 |-- long: double (nullable = true)



In [7]:
def get_unique_values(df, columns_to_check):
    print("\n--- Unique Values Report to check for inconsistencies ---")
    for col_name in columns_to_check:
        print(f"\nUnique values for {col_name}:")
        
        # 1. Get distinct values
        # 2. Drop nulls so they don't clutter the list
        # 3. Sort them so it's readable
        unique_df = df.select(col_name).distinct().na.drop().orderBy(col_name)
        
        # If the list is small, show it all. If huge, show top 20.
        unique_count = unique_df.count()
        print(f"Total Unique: {unique_count}")
        unique_df.show(truncate=False)
        
get_unique_values(df, ["manufacturer", "year", "condition", "fuel", "cylinders", "title_status", "transmission", "drive", "type"])        


--- Unique Values Report to check for inconsistencies ---

Unique values for manufacturer:
Total Unique: 41
+---------------+
|manufacturer   |
+---------------+
|acura          |
|alfa-romeo     |
|aston-martin   |
|audi           |
|bmw            |
|buick          |
|cadillac       |
|chevrolet      |
|chrysler       |
|datsun         |
|dodge          |
|ferrari        |
|fiat           |
|ford           |
|gmc            |
|harley-davidson|
|honda          |
|hyundai        |
|infiniti       |
|jaguar         |
+---------------+
only showing top 20 rows

Unique values for year:
Total Unique: 110
+----+
|year|
+----+
|1900|
|1901|
|1905|
|1913|
|1914|
|1915|
|1916|
|1917|
|1919|
|1920|
|1921|
|1922|
|1923|
|1924|
|1925|
|1926|
|1927|
|1928|
|1929|
|1930|
+----+
only showing top 20 rows

Unique values for condition:
Total Unique: 6
+---------+
|condition|
+---------+
|excellent|
|fair     |
|good     |
|like new |
|new      |
|salvage  |
+---------+


Unique values for fuel:
Total 

In [8]:
def check_price_accuracy(df):
    issues = []
    passed = True
    total_rows = df.count()
    # --- 1. Check Price <= 0 ---
    # We filter for values that are logically 0 or less
    invalid_price_count = df.filter(F.col("price") <= 2000).count()
    if invalid_price_count > 0:
        passed = False
        issues.append(f"{invalid_price_count} rows, ({(invalid_price_count/total_rows)*100:.2f}%) have price <= 2000")
    add_to_report("Accuracy", passed, issues)

check_price_accuracy(df)

In [9]:
def check_timeleness(df):
    issues = []
    passed = True
    total_rows = df.count()
    # --- 1. Check Price <= 0 ---
    # We filter for values that are logically 0 or less
    invalid_year_count = df.filter(F.col("year") > 2026).count()
    if invalid_year_count > 0:
        passed = False
        issues.append(f"{invalid_year_count} rows, ({(invalid_year_count/total_rows)*100:.2f}%) have year > 2026")
    add_to_report("Timeleness", passed, issues)

check_timeleness(df)

In [10]:
def audit_other_entries(df, columns_list):
    """
    Scans a list of columns for missing values and 'other' entries 
    using a single-pass aggregation for big data efficiency.
    """
    total_rows = df.count()
    if total_rows == 0:
        print("Dataset is empty.")
        return

    # 1. Build the aggregation expressions
    # We look for standard NULLs, empty strings, and the specific string 'other'
    agg_exprs = []
    for col_name in columns_list:
        # Standard Nulls / Empty
        is_null_cond = F.col(col_name).isNull() | (F.col(col_name) == "") | (F.col(col_name) == "NULL")
        agg_exprs.append(F.count(F.when(is_null_cond, 1)).alias(f"{col_name}_nulls"))
        
        # 'Other' entries (case-insensitive check)
        is_other_cond = F.lower(F.col(col_name)) == "other"
        agg_exprs.append(F.count(F.when(is_other_cond, 1)).alias(f"{col_name}_others"))

    # 2. Execute the scan once
    print(f"Auditing {len(columns_list)} columns across {total_rows} rows...")
    results = df.select(agg_exprs).collect()[0].asDict()

    # 3. Print the report
    print("\n" + "="*75)
    print(f"{'Column':<20} | {'Null %':<12} | {'Other %':<12} | {'Total Dirty %'}")
    print("-" * 75)

    for col_name in columns_list:
        null_count = results[f"{col_name}_nulls"]
        other_count = results[f"{col_name}_others"]
        
        null_pct = (null_count / total_rows) * 100
        other_pct = (other_count / total_rows) * 100
        total_dirty_pct = ((null_count + other_count) / total_rows) * 100

        print(f"{col_name:<20} | {null_pct:>6.2f}%      | {other_pct:>6.2f}%      | {total_dirty_pct:>6.2f}%")
    print("="*75)

columns_to_audit = ["manufacturer", "year", "fuel", "condition", "cylinders", "title_status", "transmission", "drive", "type", "city" ]
audit_other_entries(df, columns_to_audit)

Auditing 10 columns across 525839 rows...

Column               | Null %       | Other %      | Total Dirty %
---------------------------------------------------------------------------
manufacturer         |   4.67%      |   0.00%      |   4.67%
year                 |   0.27%      |   0.00%      |   0.27%
fuel                 |   0.82%      |   3.13%      |   3.95%
condition            |  46.77%      |   0.00%      |  46.77%
cylinders            |  40.01%      |   0.29%      |  40.30%
title_status         |   0.54%      |   0.00%      |   0.54%
transmission         |   0.81%      |   2.75%      |   3.56%
drive                |  28.79%      |   0.00%      |  28.79%
type                 |  28.32%      |   2.62%      |  30.94%
city                 |   0.00%      |   0.00%      |   0.00%


In [11]:
def analyze_outliers_and_plot(df, numeric_cols, cleaned=False):
    issues = []
    passed = True
    total_rows = df.count()
    if total_rows == 0:
        print("Dataset is empty.")
        return

    for col_name in numeric_cols:
        print("\n" + "="*50)
        print(f"Analyzing Column: {col_name}")
        print("="*50)
        
        # --- 1. Calculate IQR using approxQuantile ---
        # Parameters: (column_name, [probabilities], relative_error)
        # 0.01 error means it's 99% accurate but much faster than an exact sort
        quantiles = df.approxQuantile(col_name, [0.25, 0.75], 0.01)
        
        if len(quantiles) < 2:
            print(f"Not enough data in {col_name} to calculate quartiles.")
            continue
            
        q1, q3 = quantiles[0], quantiles[1]
        iqr = q3 - q1
        
        lower_bound = q1 - 1.5 * iqr
        upper_bound = q3 + 1.5 * iqr
        
        # Count the outliers
        outliers_count = df.filter(
            (F.col(col_name) < lower_bound) | (F.col(col_name) > upper_bound)
        ).count()
        
        outliers_pct = (outliers_count / total_rows) * 100
        
        print(f"Q1: {q1:.2f} | Q3: {q3:.2f} | IQR: {iqr:.2f}")
        print(f"Valid Range: [{lower_bound:.2f} to {upper_bound:.2f}]")
        print(f"Outliers Found: {outliers_count} rows ({outliers_pct:.2f}%)")

        if outliers_pct > 0:
            issues.append(f"{outliers_count} outliers in {col_name} ({outliers_pct:.2f}%)")
            passed = False

        # --- 2. Build the Big Data Histogram ---
        print(f"Generating histogram for {col_name}...")
        
        # We must drop NULLs and cast to Double for the RDD histogram function to work
        clean_df = df.filter(F.col(col_name).isNotNull()).select(F.col(col_name).cast("double"))
        
        try:
            # Spark calculates the bins and counts distributedly! 
            # 20 represents the number of bins we want.
            rdd_data = clean_df.rdd.flatMap(lambda x: x)
            bins, counts = rdd_data.histogram(20)
            
            # Plotting the results using Matplotlib
            fig, ax = plt.subplots(figsize=(10, 5))
            
            # Calculate the width of each bin for plotting
            bin_widths = [bins[i+1] - bins[i] for i in range(len(bins)-1)]
            
            # Draw the bar chart using the pre-aggregated Spark data
            ax.bar(bins[:-1], counts, width=bin_widths, align='edge', edgecolor='black', alpha=0.75)
            
            ax.set_title(f"Distribution of {col_name}")
            ax.set_xlabel(col_name)
            ax.set_ylabel("Frequency")
            
            # Use tight_layout to ensure labels aren't cut off
            plt.tight_layout()
            
            # Save the plot (you can also use plt.show() if running in a notebook)
            if cleaned:
                file_name = f"Reports/Figures/histogram_{col_name}_cleaned.png"
            else:
                file_name = f"Reports/Figures/histogram_{col_name}.png"
            plt.savefig(file_name)
            print(f"✅ Histogram saved as {file_name}")
            
            # Clear the plot from memory so they don't overlap
            plt.close(fig)
            
        except Exception as e:
            print(f"❌ Failed to generate histogram for {col_name}: {e}")
    
    add_to_report(f"Outliers", passed, issues)

numeric_columns_to_check = ["price", "odometer", "lat", "long"]
analyze_outliers_and_plot(df_casted, numeric_columns_to_check)


Analyzing Column: price
Q1: 3800.00 | Q3: 17900.00 | IQR: 14100.00
Valid Range: [-17350.00 to 39050.00]
Outliers Found: 16304 rows (3.10%)
Generating histogram for price...
✅ Histogram saved as Reports/Figures/histogram_price.png

Analyzing Column: odometer
Q1: 48085.00 | Q3: 137815.00 | IQR: 89730.00
Valid Range: [-86510.00 to 272410.00]
Outliers Found: 4408 rows (0.84%)
Generating histogram for odometer...
✅ Histogram saved as Reports/Figures/histogram_odometer.png

Analyzing Column: lat
Q1: 34.75 | Q3: 42.38 | IQR: 7.63
Valid Range: [23.31 to 53.83]
Outliers Found: 7204 rows (1.37%)
Generating histogram for lat...
✅ Histogram saved as Reports/Figures/histogram_lat.png

Analyzing Column: long
Q1: -106.69 | Q3: -80.93 | IQR: 25.75
Valid Range: [-145.32 to -42.31]
Outliers Found: 6883 rows (1.31%)
Generating histogram for long...
✅ Histogram saved as Reports/Figures/histogram_long.png


In [12]:
def plot_categorical_distributions(df, cat_cols, top_n=15, cleaned=False):
    """
    Groups by categories in Spark, handles Nulls safely, and plots.
    """
    for col_name in cat_cols:
        print(f"Generating bar plot for {col_name}...")
        
        # 1. Aggregate in Spark
        stats_df = df.groupBy(col_name).count().orderBy(F.desc("count")).limit(top_n)
        
        # 2. Convert to Pandas
        pdf = stats_df.toPandas()
        
        if pdf.empty:
            continue

        # --- THE FIX: Fill Nulls/NaNs with a string ---
        # This converts the float 'NaN' into the string 'Unknown'
        pdf[col_name] = pdf[col_name].fillna("Unknown").astype(str)

        # 3. Plot
        plt.figure(figsize=(12, 6))
        # Use the cleaned column for the Y-axis
        plt.barh(pdf[col_name], pdf['count'], color='#3498db', edgecolor='black')
        
        plt.gca().invert_yaxis()
        plt.title(f"Top {top_n} {col_name} Distribution")
        plt.xlabel("Frequency")
        plt.ylabel(col_name.capitalize())
        plt.tight_layout()
        if cleaned:
            plt.savefig(f"Reports/Figures/barplot_{col_name}_cleaned.png")
        else:
            plt.savefig(f"Reports/Figures/barplot_{col_name}.png")
        plt.close()
        print(f"✅ Barplot saved for {col_name}")


categorical_to_plot = ["manufacturer", "condition", "fuel", "transmission", "type","cylinders", "title_status", "drive"]
plot_categorical_distributions(df, categorical_to_plot)

Generating bar plot for manufacturer...
✅ Barplot saved for manufacturer
Generating bar plot for condition...
✅ Barplot saved for condition
Generating bar plot for fuel...
✅ Barplot saved for fuel
Generating bar plot for transmission...
✅ Barplot saved for transmission
Generating bar plot for type...
✅ Barplot saved for type
Generating bar plot for cylinders...
✅ Barplot saved for cylinders
Generating bar plot for title_status...
✅ Barplot saved for title_status
Generating bar plot for drive...
✅ Barplot saved for drive


In [13]:
def generate_report(json_path="validation_report.json"):
        """Saves validation results to JSON and CSV and returns summary statistics."""
        
        report_data = {
            "metadata": metadata,
            "results": validation_results
        }

        with open(json_path, 'w') as f:
            json.dump(report_data, f, indent=4)


        # 3. Print Summary to Terminal
        total = len(validation_results)
        passed = sum(1 for r in validation_results if r['passed'])
        
        print(f"\n{'='*55}\n VALIDATION COMPLETE\n{'='*55}")
        print(f"Files Saved: {json_path}")
        print(f"Checks: {total} | Passed: {passed} | Failed: {total - passed}")
        print(f"{'='*55}\n")

        return report_data

generate_report("Reports/Results/validation_report.json")


 VALIDATION COMPLETE
Files Saved: Reports/Results/validation_report.json
Checks: 5 | Passed: 2 | Failed: 3



{'metadata': {'total_rows': 525839,
  'total_columns': 22,
  'scan_time': '2026-05-01 20:57:17'},
 'results': [{'timestamp': '2026-05-01 20:57:30',
   'check_type': 'Completeness',
   'passed': False,
   'issues': ["Column 'year': 0.27% missing",
    "Column 'manufacturer': 4.67% missing",
    "Column 'make': 1.64% missing",
    "Column 'condition': 46.77% missing",
    "Column 'cylinders': 40.01% missing",
    "Column 'fuel': 0.82% missing",
    "Column 'odometer': 18.75% missing",
    "Column 'title_status': 0.54% missing",
    "Column 'transmission': 0.81% missing",
    "Column 'VIN': 41.86% missing",
    "Column 'drive': 28.79% missing",
    "Column 'size': 66.81% missing",
    "Column 'type': 28.32% missing",
    "Column 'paint_color': 32.62% missing",
    "Column 'image_url': 0.00% missing",
    "Column 'desc': 0.00% missing",
    "Column 'lat': 2.32% missing",
    "Column 'long': 2.32% missing"]},
  {'timestamp': '2026-05-01 20:57:36',
   'check_type': 'Schema & Types',
   'pass

## After Cleaning

In [14]:
df_cleaned = spark.read.csv("data/interim/used_cars_cleaned.csv", header=True, inferSchema=True)
validation_results = []
metadata = {
            'total_rows': df.count(),
            'total_columns': len(df.columns),
            'scan_time': datetime.now().strftime("%Y-%m-%d %H:%M:%S")
}

In [15]:
check_completeness(df_cleaned)

Total Rows: 415154
Analyzing Completeness


In [16]:
schema = StructType([
    StructField("url", StringType(), True),
    StructField("city", StringType(), True),
    StructField("price", LongType(), True),
    StructField("year", IntegerType(), True),
    StructField("manufacturer", StringType(), True),
    StructField("condition", StringType(), True),
    StructField("cylinders_numeric", IntegerType(), True),
    StructField("fuel", StringType(), True),
    StructField("odometer", DoubleType(), True),
    StructField("title_status", StringType(), True),
    StructField("transmission", StringType(), True),
    StructField("drive", StringType(), True),
    StructField("type", StringType(), True),
    StructField("long", DoubleType(), True),
    StructField("lat", DoubleType(), True)
])
check_schema_and_types(df_cleaned, schema)


--- Metadata & Type Conformance Check ---


In [17]:
check_price_accuracy(df_cleaned)

In [18]:
check_timeleness(df_cleaned)

In [19]:
columns_to_audit = ["manufacturer", "year", "fuel", "condition", "cylinders_numeric", "title_status", "transmission", "drive", "type", "city" ]
audit_other_entries(df_cleaned, columns_to_audit)

Auditing 10 columns across 415154 rows...

Column               | Null %       | Other %      | Total Dirty %
---------------------------------------------------------------------------
manufacturer         |   0.00%      |   0.00%      |   0.00%
year                 |   0.00%      |   0.00%      |   0.00%
fuel                 |   0.00%      |   0.00%      |   0.00%
condition            |   0.00%      |   0.00%      |   0.00%
cylinders_numeric    |   0.00%      |   0.00%      |   0.00%
title_status         |   0.00%      |   0.00%      |   0.00%
transmission         |   0.00%      |   0.00%      |   0.00%
drive                |   0.00%      |   0.00%      |   0.00%
type                 |   0.00%      |   0.00%      |   0.00%
city                 |   0.00%      |   0.00%      |   0.00%


In [20]:
numeric_columns_to_check = ["price", "odometer", "lat", "long"]
analyze_outliers_and_plot(df_cleaned, numeric_columns_to_check, cleaned=True)


Analyzing Column: price
Q1: 5995.00 | Q3: 18995.00 | IQR: 13000.00
Valid Range: [-13505.00 to 38495.00]
Outliers Found: 16235 rows (3.91%)
Generating histogram for price...
✅ Histogram saved as Reports/Figures/histogram_price_cleaned.png

Analyzing Column: odometer
Q1: 63417.00 | Q3: 127761.00 | IQR: 64344.00
Valid Range: [-33099.00 to 224277.00]
Outliers Found: 10267 rows (2.47%)
Generating histogram for odometer...
✅ Histogram saved as Reports/Figures/histogram_odometer_cleaned.png

Analyzing Column: lat
Q1: 34.83 | Q3: 42.43 | IQR: 7.60
Valid Range: [23.43 to 53.84]
Outliers Found: 2616 rows (0.63%)
Generating histogram for lat...
✅ Histogram saved as Reports/Figures/histogram_lat_cleaned.png

Analyzing Column: long
Q1: -108.49 | Q3: -81.29 | IQR: 27.21
Valid Range: [-149.30 to -40.48]
Outliers Found: 0 rows (0.00%)
Generating histogram for long...
✅ Histogram saved as Reports/Figures/histogram_long_cleaned.png


In [21]:
categorical_to_plot = ["manufacturer", "condition", "fuel", "transmission", "type", "cylinders_numeric", "title_status", "drive"]
plot_categorical_distributions(df_cleaned, categorical_to_plot, cleaned=True)

Generating bar plot for manufacturer...
✅ Barplot saved for manufacturer
Generating bar plot for condition...
✅ Barplot saved for condition
Generating bar plot for fuel...
✅ Barplot saved for fuel
Generating bar plot for transmission...
✅ Barplot saved for transmission
Generating bar plot for type...
✅ Barplot saved for type
Generating bar plot for cylinders_numeric...
✅ Barplot saved for cylinders_numeric
Generating bar plot for title_status...
✅ Barplot saved for title_status
Generating bar plot for drive...
✅ Barplot saved for drive


In [22]:
generate_report("Reports/Results/validation_report_cleaned.json")


 VALIDATION COMPLETE
Files Saved: Reports/Results/validation_report_cleaned.json
Checks: 5 | Passed: 3 | Failed: 2



{'metadata': {'total_rows': 525839,
  'total_columns': 22,
  'scan_time': '2026-05-01 21:02:09'},
 'results': [{'timestamp': '2026-05-01 21:02:09',
   'check_type': 'Completeness',
   'passed': True,
   'issues': []},
  {'timestamp': '2026-05-01 21:02:10',
   'check_type': 'Schema & Types',
   'passed': False,
   'issues': []},
  {'timestamp': '2026-05-01 21:02:10',
   'check_type': 'Accuracy',
   'passed': True,
   'issues': []},
  {'timestamp': '2026-05-01 21:02:11',
   'check_type': 'Timeleness',
   'passed': True,
   'issues': []},
  {'timestamp': '2026-05-01 21:04:23',
   'check_type': 'Outliers',
   'passed': False,
   'issues': ['16235 outliers in price (3.91%)',
    '10267 outliers in odometer (2.47%)',
    '2616 outliers in lat (0.63%)']}]}